# Session 10 Practice: Full-Stack Integrated Motion Planning Capstone
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 5 | Session 10**  

---

### 🎯 Learning Objectives
1. Integrate the full autonomous vehicle motion planning hierarchy:
   - **Mission Layer:** Global corridor definition
   - **Behaviour Layer:** Hierarchical state machine (Cruising, Stop-Line Deceleration, Stop-and-Wait, Intersection Crossing)
   - **Local Motion Layer:** Reactive trajectory rollout generation and circle collision avoidance
   - **Vehicle Control & Dynamics Layer:** Kinematic bicycle forward simulation
2. Execute closed-loop navigation through dynamic urban scenarios with obstacles and stop signs.
3. Build an end-to-end telemetry dashboard with **Plotly**.

---

### 📐 System Architecture

```
+-------------------------------------------------------------+
|                     Mission Waypoints                       |
+-------------------------------------------------------------+
                              |
                              v
+-------------------------------------------------------------+
|                HSM Behaviour Planner                        |
|  [FOLLOW_LANE] -> [APPROACH_STOP] -> [WAIT] -> [CROSS]      |
+-------------------------------------------------------------+
                              | Target Speed & Maneuver State
                              v
+-------------------------------------------------------------+
|           Local Rollout & Collision Checker                 |
|  Circle Footprint Swath Collision Checking vs Obstacles     |
+-------------------------------------------------------------+
                              | Optimal Steering (delta) & Accel (a)
                              v
+-------------------------------------------------------------+
|              Kinematic Bicycle Vehicle Model                |
|  [x, y, theta, v] Telemetry Logging & KPI Dashboard         |
+-------------------------------------------------------------+
```


In [ ]:
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Full-Stack Autonomous Planner Implementation

In [ ]:
class FullStackAutonomousPlanner:
    def __init__(self, stop_sign_s: float = 55.0, v_cruise: float = 9.0):
        self.stop_sign_s = stop_sign_s
        self.v_cruise = v_cruise
        
        self.state = "FOLLOW_LANE"
        self.stop_timer = 0.0
        
        self.L = 2.8
        self.max_lat_accel = 2.5
        self.collision_radius = 1.25
        self.circle_offsets = [-1.0, 0.5, 2.0]

    def plan_step(self, ego_s: float, ego_x: float, ego_y: float,
                  ego_theta: float, ego_v: float,
                  static_obstacles: list[tuple], dt: float) -> tuple[float, float, str]:
        dist_to_stop = self.stop_sign_s - ego_s
        
        # 1. Behaviour Layer
        if self.state == "FOLLOW_LANE":
            target_v = self.v_cruise
            if 0 < dist_to_stop <= 25.0:
                self.state = "APPROACH_STOP"
                
        elif self.state == "APPROACH_STOP":
            target_v = max(0.0, min(self.v_cruise, (dist_to_stop / 25.0) * self.v_cruise))
            if dist_to_stop <= 1.2 and ego_v < 0.2:
                self.state = "STAY_STOPPED"
                self.stop_timer = 0.0
                
        elif self.state == "STAY_STOPPED":
            target_v = 0.0
            self.stop_timer += dt
            if self.stop_timer >= 3.0: # 3 second wait
                self.state = "CROSS_INTERSECTION"
                
        elif self.state == "CROSS_INTERSECTION":
            target_v = 7.0
            if ego_s > self.stop_sign_s + 15.0:
                self.state = "FOLLOW_LANE"
                
        # 2. Local Rollout Selection (Avoid Static Obstacles)
        steer_candidates = np.linspace(-0.35, 0.35, 17)
        best_delta = 0.0
        best_cost = float('inf')
        
        for delta in steer_candidates:
            collides = False
            cur_x, cur_y, cur_th = ego_x, ego_y, ego_theta
            
            for _ in range(10): # 2.0s horizon (10 steps of 0.2s)
                cur_x += ego_v * math.cos(cur_th) * 0.2
                cur_y += ego_v * math.sin(cur_th) * 0.2
                cur_th += (ego_v / self.L) * math.tan(delta) * 0.2
                
                for off in self.circle_offsets:
                    cx = cur_x + off * math.cos(cur_th)
                    cy = cur_y + off * math.sin(cur_th)
                    for ox, oy, robs in static_obstacles:
                        if math.hypot(cx - ox, cy - oy) < (self.collision_radius + robs):
                            collides = True
                            break
                    if collides:
                        break
                if collides:
                    break
                    
            if not collides:
                cost = abs(cur_y - 0.0) + 0.8 * abs(delta)
                if cost < best_cost:
                    best_cost = cost
                    best_delta = delta
                    
        # 3. Longitudinal Acceleration Command
        accel_cmd = max(-3.0, min(1.5, (target_v - ego_v) * 1.2))
        
        return best_delta, accel_cmd, self.state


## 2. Closed-Loop Mission Execution Simulation

In [ ]:
def run_full_simulation(total_time: float = 18.0, dt: float = 0.05) -> pl.DataFrame:
    planner = FullStackAutonomousPlanner(stop_sign_s=55.0, v_cruise=9.0)
    
    # Parked car obstacle on right lane at x=25m, y=0m
    obstacles = [(25.0, 0.0, 1.4)]
    
    ego_s = 0.0
    ego_x = 0.0
    ego_y = 0.0
    ego_theta = 0.0
    ego_v = 4.0
    
    telemetry = []
    num_steps = int(total_time / dt)
    
    for step in range(num_steps):
        t = step * dt
        
        steer_cmd, accel_cmd, state_str = planner.plan_step(
            ego_s, ego_x, ego_y, ego_theta, ego_v, obstacles, dt
        )
        
        # Vehicle Kinematics Update
        ego_x += ego_v * math.cos(ego_theta) * dt
        ego_y += ego_v * math.sin(ego_theta) * dt
        ego_theta += (ego_v / planner.L) * math.tan(steer_cmd) * dt
        ego_v = max(0.0, ego_v + accel_cmd * dt)
        ego_s += ego_v * dt
        
        kappa = math.tan(steer_cmd) / planner.L
        a_lat = (ego_v ** 2) * kappa
        
        telemetry.append({
            "time_s": round(t, 2),
            "ego_s_m": round(ego_s, 2),
            "x_m": round(ego_x, 2),
            "y_m": round(ego_y, 2),
            "theta_rad": round(ego_theta, 3),
            "velocity_mps": round(ego_v, 2),
            "steer_rad": round(steer_cmd, 3),
            "accel_mps2": round(accel_cmd, 2),
            "lat_accel_mps2": round(a_lat, 2),
            "state": state_str
        })
        
    return pl.DataFrame(telemetry)

df_capstone = run_full_simulation(total_time=18.0, dt=0.05)
print(f"Capstone simulation completed: {len(df_capstone)} telemetry frames.")


## 3. Comprehensive End-to-End Capstone Dashboard

In [ ]:
def generate_capstone_dashboard(df: pl.DataFrame, obstacles: list[tuple]):
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "2D Mission Trajectory & Obstacle Overtaking",
            "Speed Profile v(t)",
            "Steering Angle Command δ(t)",
            "Lateral Acceleration vs Comfort Limit"
        ),
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )
    
    # 1. 2D Path
    for ox, oy, robs in obstacles:
        fig.add_shape(type="circle", xref="x", yref="y",
                      x0=ox - robs, y0=oy - robs, x1=ox + robs, y1=oy + robs,
                      fillcolor="crimson", line_color="darkred", row=1, col=1)
    fig.add_trace(go.Scatter(x=df["x_m"], y=df["y_m"], mode="lines+markers",
                             name="Vehicle Trajectory", line=dict(color="#1f77b4", width=3)), row=1, col=1)
    fig.add_vline(x=55.0, line_dash="dash", line_color="orange", annotation_text="Stop Sign", row=1, col=1)
    
    # 2. Velocity
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["velocity_mps"], mode="lines",
                             name="Velocity (m/s)", line=dict(color="#2ca02c", width=2.5)), row=1, col=2)
    
    # 3. Steering
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["steer_rad"], mode="lines",
                             name="Steering (rad)", line=dict(color="#ff7f0e", width=2.5)), row=2, col=1)
    
    # 4. Lateral Acceleration
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["lat_accel_mps2"], mode="lines",
                             name="a_lat (m/s²)", line=dict(color="#9467bd", width=2.5)), row=2, col=2)
    fig.add_hline(y=2.5, line_dash="dash", line_color="crimson", row=2, col=2)
    fig.add_hline(y=-2.5, line_dash="dash", line_color="crimson", row=2, col=2)
    
    fig.update_layout(title="Full-Stack Integrated Motion Planning Capstone Dashboard", height=750, showlegend=False)
    fig.show()

generate_capstone_dashboard(df_capstone, [(25.0, 0.0, 1.4)])


## 4. Verification Tests

In [ ]:
# Verify stop sign stop
stopped_df = df_capstone.filter(pl.col("state") == "STAY_STOPPED")
assert len(stopped_df) > 0, "Vehicle did not transition to STAY_STOPPED state!"
assert stopped_df["velocity_mps"].min() < 0.2, "Vehicle failed to achieve complete stop at line!"

# Verify obstacle clearance
obstacle_x, obstacle_y, r_obs = 25.0, 0.0, 1.4
near_obs = df_capstone.filter((pl.col("x_m") - obstacle_x).abs() < 2.0)
min_dist = math.hypot(near_obs["x_m"][0] - obstacle_x, near_obs["y_m"][0] - obstacle_y)
assert min_dist > r_obs + 0.5, f"Vehicle cut too close to obstacle ({min_dist:.2f}m)!"

print("✅ Full-Stack Autonomous Motion Planning Capstone passed all tests!")
